In [1]:
import cml.data_v1 as cmldata
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit


# Sample in-code customization of spark configurations
#from pyspark import SparkContext
#SparkContext.setSystemProperty('spark.executor.cores', '1')
#SparkContext.setSystemProperty('spark.executor.memory', '2g')

CONNECTION_NAME = "pdnd-prod-dl-1"
conn = cmldata.get_connection(CONNECTION_NAME)
spark = conn.get_spark_session()
print("Connection created")

# Sample usage to run query through spark
# EXAMPLE_SQL_QUERY = "show databases"
# spark.sql(EXAMPLE_SQL_QUERY).show()

input_file_name = "Transazioni-POS-Q2-cumulativo_v2.csv"
target_table = "pagopa_dev.pos_v2_transaction_q2"

absolute_path = os.path.abspath(input_file_name)
file_uri = f"file://{absolute_path}"


Setting spark.hadoop.yarn.resourcemanager.principal to andrea.ferracci


Spark Application Id:spark-e94290a0d65141719982af370998fa41
Connection created


In [2]:
print("Step 1: Reading the CSV file...")
df_csv = spark.read.option("header", "true").option("sep", ";").csv(file_uri)

Step 1: Reading the CSV file...


In [3]:
print("Step 2: Aligning schema and writing to Impala...")

# Add an empty PAYMENT_ID column to match the target table schema perfectly
df_csv_staged = df_csv.withColumn("PAYMENT_ID", lit(None).cast("string"))

Step 2: Aligning schema and writing to Impala...


In [4]:
# print("Step 3: Write data to the table")
# df_csv_staged.write \
#     .mode("overwrite") \
#     .insertInto(target_table)
# print(f"SUCCESS! Raw data loaded into {target_table}.")

print("Step 3: Writing data to Iceberg table...")
df_csv_staged.write \
    .format("iceberg") \
    .mode("append") \
    .save(target_table)

print("SUCCESS! Data loaded into Iceberg table.")

Step 3: Writing data to Iceberg table...


SUCCESS! Data loaded into Iceberg table.
